## AI4Climate ML tutorial - template
* Author: <INSERT AUTHOR>
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we will use the previously prepared tabular dataset to predict climate zones. We will train on different eras to see how well the results generalise with climate change. 

### Prerequisites 
what background information is needed to go through the notebook
- Same as previouis notebooks
- Have completed data exploration notebook

### Learning outcomes from completing the notebook

- Understand the key components of a machine learning training pipeline and how they fit together
- 

## Tutorial 
a balance of explanation and activity



TODO:
* for training train supervised and unsupervised (clusters to find types)
* for supervised, training with 5 and 30 class targets
* train and all and a few and compare results of current and future climate
* cosider different train test split strategies (random, by lat.lon


### Imports

In [2]:
import pathlib
import os
import datetime
import json

In [3]:
import pandas

In [4]:
import mlflow

In [5]:
import sklearn
import sklearn.preprocessing
import sklearn.tree
import torch

#### Dataset parameters

In [6]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, very cold winter',
  'Dwa': 'Cold, dry winter, hot summer',
  'Dwb': 'Cold, dry winter, warm summer',
  'Dwc': 'Cold, dry winter, cold su

In [7]:
def get_platform_dir(select_platform):
    if select_platform == 'mo_linux':
      return pathlib.Path(os.environ['DATADIR']) / 'climate_zones'
    if select_platform == 'jasmin':
        return pathlib.Path('/gws/nopw/j04/mohc_shared/dscop/') / 'climate_zones'
    print('platform not found, return generic path')
    return pathlib.Path(os.environ['HOME']) / 'climate_zones',


In [8]:
current_platform = tutorial_config['platform']

In [9]:
root_data_dir = get_platform_dir(current_platform)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/stephen.haddad/climate_zones')

In [10]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/data/users/stephen.haddad/climate_zones/ml_ready')

In [11]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [16]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

### Load data for training

In [20]:
current_res = 1.0

In [21]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/data/users/stephen.haddad/climate_zones/ml_ready/climate_zones_1p0.csv')

In [22]:
zones_df = pandas.read_csv(mlready_data_path)

In [23]:
zones_df.columns

Index(['lat', 'lon', 'precipitation_1.0_mean', 'precipitation_2.0_mean',
       'precipitation_3.0_mean', 'precipitation_4.0_mean',
       'precipitation_5.0_mean', 'precipitation_6.0_mean',
       'precipitation_7.0_mean', 'precipitation_8.0_mean',
       'precipitation_9.0_mean', 'precipitation_10.0_mean',
       'precipitation_11.0_mean', 'precipitation_12.0_mean',
       'air_temperature_1.0_mean', 'air_temperature_2.0_mean',
       'air_temperature_3.0_mean', 'air_temperature_4.0_mean',
       'air_temperature_5.0_mean', 'air_temperature_6.0_mean',
       'air_temperature_7.0_mean', 'air_temperature_8.0_mean',
       'air_temperature_9.0_mean', 'air_temperature_10.0_mean',
       'air_temperature_11.0_mean', 'air_temperature_12.0_mean',
       'precipitation_1.0_std', 'precipitation_2.0_std',
       'precipitation_3.0_std', 'precipitation_4.0_std',
       'precipitation_5.0_std', 'precipitation_6.0_std',
       'precipitation_7.0_std', 'precipitation_8.0_std',
       'precipitatio

In [24]:
test_frac = 0.2
val_frac = 0.2

In [25]:
zones_df['test'] = False
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac)
zones_df['test'][test_df.index] = True

/var/tmp/ipykernel_451183/1662537359.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zones_df['test'][test_df.index] = True


In [26]:
zones_df['val'] = False
val_df = zones_df[zones_df['test'] == False].groupby(['period_start','scenario']).sample(frac=(0.2)/(1.0-test_frac))
zones_df['val'][val_df.index] = True

/var/tmp/ipykernel_451183/2859563780.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zones_df['val'][val_df.index] = True


In [27]:
train_df = zones_df[((zones_df['val'] == False) & (zones_df['test'] == False ))]

In [28]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


We'll start by using only the climate means as predictors, and only aiming for the 5 classes, rather than the full 30. 

In [29]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

In [30]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [31]:
train_df[predictors].describe()

,precipitation_1.0_mean,precipitation_2.0_mean,precipitation_3.0_mean,precipitation_4.0_mean,precipitation_5.0_mean,precipitation_6.0_mean,precipitation_7.0_mean,precipitation_8.0_mean,precipitation_9.0_mean,precipitation_10.0_mean,...,air_temperature_3.0_mean,air_temperature_4.0_mean,air_temperature_5.0_mean,air_temperature_6.0_mean,air_temperature_7.0_mean,air_temperature_8.0_mean,air_temperature_9.0_mean,air_temperature_10.0_mean,air_temperature_11.0_mean,air_temperature_12.0_mean
count,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,...,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000,243900.000000
mean,45.828696,42.661319,47.492186,45.604829,49.273555,54.029885,62.492535,62.770942,53.875015,49.388231,...,-6.299059,-4.113796,-1.234663,1.449880,2.224018,1.520235,-0.327655,-2.078609,-3.241934,-3.893422
std,74.213410,67.404914,70.667409,64.565725,67.241138,76.910027,86.377895,80.732320,69.201460,64.244221,...,25.475690,27.258667,28.229976,29.036357,30.222021,30.156668,28.799533,25.536935,21.604172,19.924144
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-55.875000,-61.687500,-63.000000,-62.312500,-63.875000,-63.937500,-62.937500,-55.187500,-41.875000,-43.812500
25%,4.562500,5.187500,7.500000,9.062500,9.875000,6.375000,7.437500,10.625000,7.375000,8.000000,...,-26.062500,-25.375000,-25.000000,-25.625000,-27.375000,-28.062500,-26.375000,-23.125000,-21.437500,-21.375000
50%,17.125000,16.562500,20.750000,22.562500,26.375000,27.500000,35.687500,38.937500,33.125000,27.812500,...,-5.937500,1.687500,8.187500,13.562500,15.937500,14.437500,9.500000,2.750000,-5.250000,-8.812500
75%,46.750000,42.562500,50.625000,49.500000,57.812500,71.250000,81.437500,80.312500,67.812500,63.062500,...,19.687500,20.687500,21.375000,23.250000,24.312500,24.250000,23.062500,21.437500,19.187500,15.937500
max,734.437500,558.937500,649.062500,638.750000,935.125000,1237.062500,1592.937500,1154.125000,819.187500,938.187500,...,36.500000,38.937500,40.687500,42.437500,44.687500,44.187500,40.875000,37.312500,37.062500,37.125000


In [32]:
input_scaler = sklearn.preprocessing.StandardScaler()
input_scaler.fit(train_df[predictors])


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [33]:
input_scaler.mean_

array([45.82869593, 42.66131893, 47.49218583, 45.60482908, 49.27355499,
       54.02988469, 62.49253485, 62.77094224, 53.87501486, 49.38823109,
       44.8645759 , 45.60559194, -4.9089058 , -6.28367261, -6.29905878,
       -4.11379561, -1.23466328,  1.44987982,  2.22401804,  1.52023498,
       -0.32765478, -2.0786088 , -3.24193445, -3.89342225])

In [34]:
X_train = input_scaler.transform(train_df[predictors])
X_val = input_scaler.transform(val_df[predictors])
X_test = input_scaler.transform(test_df[predictors])

In [41]:
target_encoder = sklearn.preprocessing.OneHotEncoder()
target_encoder.fit(train_df[[target_var]])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_catego

In [54]:
train_df[[target_var]].value_counts()

climate_group
E                86888
D                60722
B                45459
A                30378
C                20453
Name: count, dtype: int64

In [65]:
y_train = target_encoder.transform(train_df[[target_var]]).toarray()
y_val = target_encoder.transform(val_df[[target_var]]).toarray()
y_test = target_encoder.transform(test_df[[target_var]]).toarray()


In [46]:
dt_clf = sklearn.tree.DecisionTreeClassifier(min_samples_split=5, min_samples_leaf=2, max_depth=5)
dt_clf

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [59]:
dt_clf.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [60]:
y_pred_train = dt_clf.predict(X_train)

In [63]:
sklearn.metrics.precision_recall_fscore_support(y_train, y_pred_train)

(array([0.89721514, 0.93704355, 0.87565767, 0.9589619 , 0.99690744]),
 array([0.95026006, 0.85521019, 0.88696035, 0.96207305, 0.98686815]),
 array([0.92297608, 0.89425864, 0.88127277, 0.96051496, 0.9918624 ]),
 array([30378, 45459, 20453, 60722, 86888]))

In [66]:
y_pred_val= dt_clf.predict(X_val)

In [67]:
sklearn.metrics.precision_recall_fscore_support(y_val, y_pred_val)

(array([0.89551258, 0.93453355, 0.87056428, 0.95565945, 0.99702193]),
 array([0.94821074, 0.85290297, 0.87951807, 0.96105117, 0.98747311]),
 array([0.92110854, 0.89185427, 0.87501827, 0.95834773, 0.99222455]),
 array([10060, 15398,  6806, 20206, 28818]))

In [ ]:
# train decision tree, random forest etc.

In [68]:
km_clusterer = sklearn.cluster.KMeans(n_clusters=5)
km_clusterer

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",5
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",None
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


In [69]:
km_clusterer.fit(X_train)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",5
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",None
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


In [70]:
km_clusterer.predict(X_train)a

array([3, 3, 3, ..., 2, 2, 2], shape=(243900,), dtype=int32)

In [71]:
#todo - plots of data on a map

## Exercises
for students to try that do not have solutions but maybe have an answer or benchmark to facilitate understanding


In [55]:
# train with subgroup target

In [56]:
# use different train/test split methods

In [58]:
# try to balance classes in train set

### Next steps or potential follow on material



###  Exmaples of Use


### Data statement
###     References
